# Chapter 2 — Backend with FastAPI
### ML Engineer (Production) Interview Playbook

**Topics:** FastAPI · Dependency Injection · Repository · Service · Error Handling · Logging · Config · Background Tasks

> In the ML Engineer (Production) role, the model is only a small part of the system; what makes the
> model available to users and other systems is a stable backend service. FastAPI — thanks to native
> async support, automatic validation with Pydantic, strong typing, and automatic OpenAPI docs — is the
> standard choice for serving models and building data APIs. In this chapter we build not just syntax,
> but the clean architecture and layering of a production service — exactly what a tech lead wants to
> see in your code.
>
> **Interview tip:** The core message of this chapter: **thin controller, service holds logic,
> repository holds data.** If business logic leaks into route handlers, the code becomes untestable and
> unmaintainable. This separation is the single most important thing evaluated in code review.

### Layered architecture at a glance

| Layer | Responsibility | What it must NOT contain |
|---|---|---|
| Router / Controller | Receive request, validate input, call the service | Business logic, raw database queries |
| Service | Business logic, orchestration, transaction boundary | HTTP details, raw SQL |
| Repository | Data access, entity mapping | Business logic, decision-making |
| Model / Schema | Data structure and validation | Complex behavior |

## 2.1 FastAPI and Pydantic Basics

FastAPI runs on ASGI (not the old WSGI), so it has true async support. Its three pillars: (1) automatic
validation and serialization with Pydantic, (2) built-in dependency injection, (3) automatic
OpenAPI/Swagger documentation generation. Pydantic models are used both for input (validation) and
output (serialization).

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from decimal import Decimal

app = FastAPI(title="Fraud Scoring API")

class ScoreRequest(BaseModel):
    user_id: str = Field(min_length=1)
    amount: Decimal = Field(gt=0)               # validation: must be positive
    currency: str = Field(pattern="^[A-Z]{3}$")

class ScoreResponse(BaseModel):
    user_id: str
    risk: float = Field(ge=0, le=1)

@app.post("/score", response_model=ScoreResponse)
async def score(req: ScoreRequest) -> ScoreResponse:
    # req is already validated; only business logic goes here
    return ScoreResponse(user_id=req.user_id, risk=0.87)


**Note:** Why `Decimal` for money and not `float`? `float` has floating-point rounding errors
(`0.1 + 0.2 != 0.3`). In financial systems, always use `Decimal` or integer cents. This question is
common at companies like bunq.

### Managing the lifecycle with `lifespan`

Loading a model or building a connection pool must happen once at startup, not on every request. The
modern approach (replacing the deprecated `on_event`) is to use `lifespan`.

In [ ]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup
    app.state.model = load_model(settings.model_path)
    app.state.db_pool = await create_pool(settings.database_url)
    yield
    # shutdown
    await app.state.db_pool.close()

app = FastAPI(lifespan=lifespan)


### `async def` vs. `def` in route handlers

If a handler does async I/O (an async query, an HTTP call), write it as `async def`. If the function is
blocking and sync (e.g. inference from a CPU-bound model or a sync library), and you declare it as
`async def` and it blocks, the **entire event loop stalls**. Fix: declare it as a regular `def` so
FastAPI runs it in a thread pool, or explicitly offload it to an executor.

In [ ]:
# Async I/O -> async def
@app.get("/user/{uid}")
async def get_user(uid: str):
    return await repo.get(uid)

# Heavy sync work -> def (FastAPI runs it in a thread pool)
@app.post("/predict")
def predict(req: PredictRequest):
    return model.predict(req.features)  # blocking, but runs in a thread pool


### Q2.1 — Why do you prefer FastAPI over Flask?

Several practical reasons: (1) native async support, essential for I/O-bound services with high
concurrency. (2) Automatic validation and serialization with Pydantic, which reduces boilerplate and
bugs. (3) Type hints as a first-class part of the framework, which tooling and IDEs benefit from.
(4) Automatic OpenAPI/Swagger docs. Flask is simple and mature, but needs extensions and manual code
for all of this. For a modern ML/data service, FastAPI gives more productivity and safety.

## 2.2 Dependency Injection

Dependency injection means dependencies (database, cache, services, current user) are supplied from
the outside rather than built inside the function. FastAPI has this natively via `Depends`. Three
benefits: testability (inject a mock), managing resource lifetime, and eliminating repetition.

In [ ]:
from fastapi import Depends
from typing import Annotated

async def get_db():
    async with SessionLocal() as session:
        yield session   # setup before, teardown after yield

# Chained (sub-)dependencies
async def get_user_repo(db=Depends(get_db)) -> UserRepository:
    return SqlUserRepository(db)

# Modern syntax with Annotated
RepoDep = Annotated[UserRepository, Depends(get_user_repo)]

@app.get("/users/{uid}")
async def read_user(uid: str, repo: RepoDep):
    return await repo.get(uid)


**Interview tip:** A dependency with `yield` behaves like a context manager: the code before
`yield` runs on entry, and the code after it (even on error) runs on exit. This guarantees the database
connection is always closed.

### Dependency caching within a single request

If a dependency is needed multiple times within one request's chain, FastAPI runs it only once by
default and caches it (scoped to that request). To disable this: `Depends(dep, use_cache=False)`.

### Overriding for tests

The strongest benefit of DI shows up in testing: you swap the real dependency for a test version
without changing the service code.

In [ ]:
def get_test_db():
    # in-memory database or mock
    ...

app.dependency_overrides[get_db] = get_test_db   # swap for tests

# ... run tests ...
app.dependency_overrides.clear()


### Q2.2 — How does DI simplify testing?

Because instead of a handler building its own database or external service, it receives it via
`Depends`. In tests, I use `app.dependency_overrides` to replace the real dependency with a mock or an
in-memory database, so tests run fast and deterministically without needing a real database. I can also
simulate error scenarios (e.g. database unavailable) with a mock that raises an exception.

## 2.3 Repository Pattern

A repository is a layer that hides data access behind an interface. Business logic doesn't know
whether data comes from Postgres, Redis, or an external API — it only works against the contract. This
separation enables swapping data sources, testing, and honoring the Dependency Inversion principle.

In [ ]:
from typing import Protocol
from dataclasses import dataclass

@dataclass
class User:
    id: str
    country: str
    risk_flag: bool

class UserRepository(Protocol):
    async def get(self, uid: str) -> User | None: ...
    async def save(self, user: User) -> None: ...

class SqlUserRepository:
    def __init__(self, session):
        self._session = session

    async def get(self, uid: str) -> User | None:
        row = await self._session.get(UserRow, uid)
        return User(row.id, row.country, row.risk_flag) if row else None

    async def save(self, user: User) -> None:
        await self._session.merge(UserRow(**user.__dict__))


**Interview tip:** Design note: the repository should return a domain entity (`User`), not a raw
ORM row (`UserRow`). This mapping keeps the domain layer independent of database details — if you swap
ORMs tomorrow, business logic remains untouched.

### Unit of Work

Sometimes several repositories need to commit within a single transaction (e.g. debiting one account
and crediting another). The Unit of Work pattern manages the transaction boundary so either everything
succeeds together or nothing does.

In [ ]:
class UnitOfWork:
    def __init__(self, session_factory):
        self._factory = session_factory

    async def __aenter__(self):
        self._session = self._factory()
        self.users = SqlUserRepository(self._session)
        self.accounts = SqlAccountRepository(self._session)
        return self

    async def __aexit__(self, exc_type, *args):
        if exc_type:
            await self._session.rollback()
        else:
            await self._session.commit()
        await self._session.close()

async with UnitOfWork(session_factory) as uow:
    await uow.accounts.debit("a1", 100)
    await uow.accounts.credit("a2", 100)   # both in one transaction


### Q2.3 — What's the difference between Repository and DAO?

Both encapsulate data access, but their focus differs. A DAO (Data Access Object) is close to the table
and CRUD operations, usually mapped one-to-one with a table. A Repository is a higher-level,
domain-driven concept: it presents a collection of domain entities as if it were an in-memory
collection, and can combine multiple data sources. In practice the line sometimes blurs, but Repository
emphasizes domain language while DAO emphasizes table structure.

## 2.4 Service Pattern

The Service layer holds business logic and coordinates repositories for data access. The controller
(route) only takes input and delegates to the service; decision-making, business rules, and the
transaction boundary live here. This makes business logic independent of HTTP and testable.

In [ ]:
class FraudNotFound(Exception): ...

class FraudService:
    def __init__(self, users: UserRepository, model, cache):
        self._users = users
        self._model = model
        self._cache = cache

    async def evaluate(self, uid: str, amount: Decimal) -> float:
        # 1) cache
        if (cached := await self._cache.get(uid)) is not None:
            return cached

        # 2) data
        user = await self._users.get(uid)
        if user is None:
            raise FraudNotFound(uid)

        # 3) business logic
        features = build_features(user, amount)
        risk = self._model.predict(features)

        # 4) side effect
        await self._cache.set(uid, risk, ttl=300)
        return risk


**Interview tip:** The controller should know nothing about this logic. The handler should be
only: `result = await service.evaluate(...)` and translate it into an HTTP response. If you see a
business `if` inside a handler, it probably belongs in the service.

### Q2.4 — Where should business logic live, and why not in the controller?

Business logic should live in the service layer, not the controller. Reasons: (1) The controller is
tied to HTTP; logic in the service is independent of HTTP and testable/reusable (e.g. from a worker or
CLI). (2) Thin controllers are more readable and have a single responsibility. (3) Business rules stay
centralized in one place instead of scattered across endpoints. The controller is only an HTTP↔domain
translator: validate input and hand it to the service, map the service's output or error to an HTTP
response.

## 2.5 Error Handling

In production, errors must be structured, meaningful, and safe. A good pattern: the domain layer raises
its own meaningful exceptions (with no knowledge of HTTP), and a central mapping layer converts them
into status codes and a uniform response.

In [ ]:
from fastapi import Request
from fastapi.responses import JSONResponse

# Domain errors -- independent of HTTP
class DomainError(Exception):
    code = "domain_error"; status = 400

class UserNotFound(DomainError):
    code = "user_not_found"; status = 404
    def __init__(self, uid): self.uid = uid

@app.exception_handler(DomainError)
async def handle_domain(request: Request, exc: DomainError):
    return JSONResponse(
        status_code=exc.status,
        content={"error": exc.code, "request_id": request.state.request_id},
    )


**Note:** Never return internal details (stack trace, raw database error text, SQL query) to the
client — it's both a security risk and an information leak. Pattern: log the full details internally,
return a generic message and a stable error code to the client.

### Validation errors (422)

FastAPI/Pydantic automatically returns a 422 with details of the invalid fields for bad input. You can
normalize this response with a custom handler so it's consistent with the rest of the API's errors.

### Q2.5 — How do you handle errors in a large API?

I define a hierarchy of domain exceptions with no dependency on HTTP (e.g. `UserNotFound`,
`InsufficientBalance`). Services raise these. Central exception handlers in the web layer then map them
to status codes and a uniform error body (including a stable error code and `request_id`). Unexpected
errors map to a generic 500 with full internal logging. This separation keeps logic clean and responses
reliable for clients.

## 2.6 Logging and Observability

In a distributed system, logs are your eyes. Structured logging (JSON) is essential so tools like ELK,
Datadog, or Grafana Loki can search and filter it. The most important principle: a **correlation id**
(request id) that makes a single request traceable across services and logs.

In [ ]:
import logging, json, uuid
from contextvars import ContextVar

request_id_ctx: ContextVar[str] = ContextVar("request_id", default="-")

class JsonFormatter(logging.Formatter):
    def format(self, record):
        return json.dumps({
            "level": record.levelname,
            "msg": record.getMessage(),
            "request_id": request_id_ctx.get(),
            "logger": record.name,
        })

# Middleware that assigns an id to every request
@app.middleware("http")
async def add_request_id(request, call_next):
    rid = request.headers.get("X-Request-ID", str(uuid.uuid4()))
    request_id_ctx.set(rid)
    request.state.request_id = rid
    response = await call_next(request)
    response.headers["X-Request-ID"] = rid
    return response


**Note:** What to never log: passwords, tokens, card numbers, sensitive PII. In financial systems
this is a legal requirement (GDPR), not just a style preference. Mask sensitive data before logging it.

**The three pillars of observability:** logs (discrete events), metrics (aggregated numbers like
latency and error rate), and traces (a request's path across services). A production service needs all
three.

### Q2.6 — What is a request id / correlation id, and why does it matter?

A unique identifier assigned to every incoming request and propagated through all logs and downstream
calls (to the database, queues, other services). Its importance: in a distributed system, one operation
might produce logs across ten different services; with a correlation id you can gather all the logs for
one specific request together and trace the problem. It's typically generated by middleware and
propagated through a header (e.g. `X-Request-ID`) and a context variable.

## 2.7 Config and Configuration Management

Per the 12-Factor App principle, configuration should be separated from code and read from environment
variables, not hardcoded. This allows a single image to run in different environments
(dev/staging/prod) with different settings. Pydantic Settings does this cleanly, with validation.

In [ ]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field

class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=".env", env_prefix="APP_")

    database_url: str
    redis_url: str
    model_path: str = "/models/latest.pkl"
    log_level: str = "INFO"
    max_batch_size: int = Field(default=32, gt=0)

settings = Settings()   # errors at startup if a required variable is missing (fail fast)


**Interview tip:** Secrets like database passwords must never live in an `.env` file inside the
code repository. In production, use a secret manager (e.g. AWS Secrets Manager, Vault) or Kubernetes
Secrets, and inject them as env vars.

### Q2.7 — Why put config in environment variables instead of code?

Several reasons: (1) Separating code from configuration means a single image/binary runs in every
environment, with only the env changing — this makes deploys reliable and repeatable. (2) Secrets stay
out of the code repository (security). (3) Changing configuration doesn't require a rebuild. This is
exactly the "config" principle of the 12-Factor App. With Pydantic Settings I validate values and fail
very early at startup if something is missing (fail fast).

## 2.8 Background Tasks and Async Work

Work that shouldn't delay the user's response (sending an email, heavy audit logging, calling a
webhook) should run in the background. FastAPI has `BackgroundTasks` for lightweight jobs, but it's
in-process and gives no execution guarantee.

In [ ]:
from fastapi import BackgroundTasks

@app.post("/score")
async def score(req: ScoreRequest, bg: BackgroundTasks):
    risk = await service.evaluate(req.user_id, req.amount)
    bg.add_task(persist_audit, req.user_id, risk)   # runs after the response is sent
    return {"risk": risk}


| Feature | BackgroundTasks | Durable queue (Celery / Kafka) |
|---|---|---|
| Where it runs | Same process | Separate worker |
| If the process crashes | The job is lost | The job survives |
| Delivery guarantee | None | At-least-once |
| Retry and scheduling | No | Yes |
| Best for | Light, non-critical work | Critical, heavy, distributed work |

**Note:** Rule of thumb: if losing the job is acceptable (e.g. an optional log entry), `BackgroundTasks`
is enough. If the job is critical (e.g. processing a payment or guaranteeing a notification is sent),
use a durable queue with delivery guarantees and retry. This distinction matters in interviews.

In ML architectures, queues play a key role for batch inference, scheduled retraining, and data
pipelines — this is covered more deeply in the "Distributed Systems" chapter.

### Q2.8 — When do you use BackgroundTasks vs. Celery?

I use `BackgroundTasks` for light, short, non-critical work that runs in the same process and where
losing it isn't a disaster (e.g. recording a metric or sending a low-importance email). I use Celery
(or a durable queue like Kafka/RabbitMQ + workers) when: the work is heavy or long-running, it needs
execution guarantees (at-least-once) and retry, it needs to scale across multiple workers, or it needs
to be scheduled. The core difference: `BackgroundTasks` loses the job if the process crashes; a durable
queue preserves it.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Architecture:** Thin controller → service (logic) → repository (data). Business logic never lives
  in a handler.
- **FastAPI:** ASGI and native async; Pydantic for validation; `lifespan` for loading models/pools;
  declare heavy sync work as `def` so it runs in a thread pool.
- **DI:** `Depends`; a `yield`-based dependency for resource management; `dependency_overrides` for
  tests.
- **Repository:** Hide data access behind a `Protocol`; return domain entities, not ORM rows; Unit of
  Work for multi-repository transactions.
- **Service:** Home of business logic and the transaction boundary; independent of HTTP and testable.
- **Errors:** Domain exceptions + a central handler; never leak internal details to the client; 422 for
  validation.
- **Logging:** Structured JSON + a request id from middleware; never log PII/secrets; three pillars:
  log, metric, trace.
- **Config:** Pydantic Settings from env (12-Factor); secrets in a secret manager; fail fast at
  startup.
- **Background:** `BackgroundTasks` is light and unguaranteed; use a durable queue with retry and
  at-least-once delivery for critical work.